# Vegetation challenge: data exploration

Step 1: load the files and see what we have. Keep printed output to summaries (counts, column names). Do not print raw rows, because the data are not published.

In [80]:
from pathlib import Path

import pandas as pd

## Paths

The data live outside the repository. Change `data_dir` if the files move or you are someone reproducing this workflow from your own machine.

In [81]:
data_dir = Path("/Users/scohen/Documents/NaturalState/Take Home Project/Data Vegetation")

exports_dir = data_dir / "ODK Data Exports"
entities_dir = data_dir / "Entity lists"

print(exports_dir.exists(), entities_dir.exists())

True True


## Read the files

The ODK exports are the data to check. The entity lists are the reference data.

In [82]:
exports = {p.stem: pd.read_csv(p) for p in sorted(exports_dir.glob("*.csv"))}
entities = {p.stem: pd.read_csv(p) for p in sorted(entities_dir.glob("*.csv"))}

## Overview: rows and columns per file

In [83]:
def overview(tables):
    return pd.DataFrame(
        {"rows": {name: df.shape[0] for name, df in tables.items()},
         "columns": {name: df.shape[1] for name, df in tables.items()}}
    )

overview(exports)

,rows,columns
herbaceous_veg_survey-additional_species_repeat,1296,30
herbaceous_veg_survey-quadrat_repeat,640,21
herbaceous_veg_survey,32,52
register_vegetation_plots,31,82


In [84]:
overview(entities)

,rows,columns
areas,1,10
centroids,51,21
ecosystem_types,5,10
project_team,6,13
projects,1,11
species,370,21
species_extra,250,14
species_extra_ids,109,11
surveys,4,15
transport,5,10


### Load data tables

In [85]:
for name, df in {**exports, **entities}.items():
    globals()[name.replace("-", "_")] = df

### herbaceous_veg_survey

This is the main survey file. Each row is one survey submission: a field team's visit to one registered vegetation plot, recorded in ODK using the herbaceous vegetation form. The columns describe who did the survey (team and recorder), when, which plot and survey it belongs to, how many quadrats were recorded, and where the submission was made. The species found in each of the 20 quadrats are in the linked `quadrat_repeat` and `additional_species_repeat` files, which join back to this file through `KEY`. The data are used to assess herbaceous species richness and diversity.

In [86]:
form = pd.read_excel(data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx", sheet_name="survey")
form = form.dropna(subset=["type"])

# Walk through the form, building each question's column name from its groups
groups, in_repeat, rows = [], 0, []
for _, q in form.iterrows():
    kind = str(q["type"]).strip()
    if kind == "begin_group":
        groups.append(q["name"])
    elif kind == "end_group":
        groups.pop()
    elif kind == "begin_repeat":
        in_repeat += 1
    elif kind == "end_repeat":
        in_repeat -= 1
    elif in_repeat == 0:          # questions inside repeats belong to the other files
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [q["name"]]), "form_type": kind,
                     "description": str(text)[:100]})
form_cols = pd.DataFrame(rows)

# Location questions are exported as four columns
geo = form_cols[form_cols["form_type"].isin(["geopoint", "background-geopoint"])]
form_cols = pd.concat([form_cols] + [geo.assign(column=geo["column"] + "-" + s)
                      for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

survey = herbaceous_veg_survey
schema = (pd.DataFrame({"column": survey.columns, "dtype": survey.dtypes.astype(str).values,
                        "missing": survey.isna().sum().values, "distinct": survey.nunique().values})
          .merge(form_cols, on="column", how="left"))
schema["form_type"] = schema["form_type"].fillna("ODK system column")
schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,SubmissionDate,str,0,32,ODK system column,NaN
1,note_overview,float64,32,0,note,This form is to be used by field data teams to...
2,field_team_specifics-selected_transport_type_uuid,str,0,1,select_one transport.csv,Select the the mode of transport:
3,field_team_specifics-transport_type_label,str,0,1,calculate,"calculated: instance(""transport"")/root/item[na..."
4,field_team_specifics-selected_project_team_uui...,str,0,2,select_multiple_from_file project_team.csv,Select all the team members involved in this c...
5,field_team_specifics-recorder_uuid,str,0,2,select_one_from_file project_team.csv,Who is recording the data:
6,field_team_specifics-recorder_label,str,0,2,calculate,"calculated: instance(""project_team"")/root/item..."
7,survey_begin-start_time,str,0,32,start,Start time
8,survey_begin-survey_year,int64,0,1,calculate,"calculated: format-date(${start_time}, '%Y')"
9,survey_begin-selected_survey_uuid,str,0,1,select_one_from_file surveys.csv,Select a Survey:


### Columns kept for the checks

`herbaceous_veg_survey` has 52 columns. I kept 22 of them: the ones needed to check who surveyed which plot, when, how many quadrats, where, and whether the submission is complete.

**What I kept, and the checks each group supports**

| Group | Why it is kept |
|---|---|
| Keys and links (`KEY`, plot columns) | Find repeated plot visits and confirm the plot is registered and viable |
| Team (recorder and team members) | Confirm the recorder is a valid team member |
| Timing (`SubmissionDate`, start and end time) | Check dates and survey duration |
| Survey context (survey, year, replicate) | Check the survey matches the plot's assigned survey |
| Sampling effort (quadrat count, quadrats with species) | Check for exactly 20 quadrats per plot |
| Location (latitude, longitude, accuracy) | Check the location is present, accurate, and near the plot |
| Completeness (attachments, review state, confirmation) | Check for missing photos and review flags |

**What I dropped**

- **Columns that never vary.** The project, area, survey type, target group, and transport labels each have a single value in all 32 rows, so they cannot show a problem.
- **Notes that are always empty.** The `note_*` and `quadrat_check_*` columns and the other note and heading fields are blank in every row.
- **ODK internals.** `base_active_herb_count`, `new_unknown_index_list`, `new_missing_index_list`, `meta-*`, `SubmitterID`, `Edits`, `FormVersion` and `Status` are system or calculation columns, not field observations. `DeviceID` is also dropped but could be kept to compare devices.

**Two things I noticed while choosing, to check later**

- There are 32 rows but only 30 distinct plots, so some plots were surveyed more than once.
- 3 submissions have no location (`background_geopoint` is empty).

In [87]:
keep = [
    "KEY", "plot_selection-selected_plot_uuid", "plot_selection-plot_name", "plot_selection-get_plot_status",
    "field_team_specifics-selected_project_team_uuid_multi", "field_team_specifics-recorder_uuid", "field_team_specifics-recorder_label",
    "SubmissionDate", "survey_begin-start_time", "survey_end-end_time",
    "survey_begin-selected_survey_uuid", "survey_begin-survey_year", "survey_begin-replicate",
    "observations-quadrat_repeat_count", "survey_end-quadrats_with_species",
    "survey_end-background_geopoint-Latitude", "survey_end-background_geopoint-Longitude", "survey_end-background_geopoint-Accuracy",
    "AttachmentsPresent", "AttachmentsExpected", "ReviewState", "survey_end-confirm_submission",
]

herbaceous_veg_survey_keep = herbaceous_veg_survey[keep]

### herbaceous_veg_survey-quadrat_repeat

This file holds the quadrat records. Each row is one quadrat (a 1 m × 1 m frame placed along the plot's transect) within one survey submission. Each survey should have 20 quadrats, so the 32 submissions give 640 rows. For each quadrat, the field team recorded whether any herbaceous species were present, which species from the approved list were selected, whether additional species were present, a photo, the GPS location, and an optional comment. It links to its survey through `PARENT_KEY` (matching the survey's `KEY`), and its species selections link to the `species` entity list. The species recorded here are the basis for assessing species richness and diversity.


In [88]:
def form_columns(form_path, repeat=None):
    form = pd.read_excel(form_path, sheet_name="survey").dropna(subset=["type"])
    stack, rows = [], []
    for _, q in form.iterrows():
        kind, name = str(q["type"]).strip(), q["name"]
        if kind == "begin_group":
            stack.append(("group", name)); continue
        if kind == "begin_repeat":
            stack.append(("repeat", name)); continue
        if kind in ("end_group", "end_repeat"):
            stack.pop(); continue
        repeats = [i for i, (t, n) in enumerate(stack) if t == "repeat"]
        innermost = stack[repeats[-1]][1] if repeats else None
        if innermost != repeat:
            continue
        start = repeats[-1] + 1 if repeats else 0
        groups = [n for t, n in stack[start:] if t == "group"]
        text = q["label"] if pd.notna(q["label"]) else "calculated: " + str(q["calculation"])
        rows.append({"column": "-".join(groups + [name]), "form_type": kind, "description": str(text)[:100]})
    cols = pd.DataFrame(rows)
    geo = cols[cols["form_type"].isin(["geopoint", "background-geopoint"])]
    return pd.concat([cols] + [geo.assign(column=geo["column"] + "-" + s)
                     for s in ["Latitude", "Longitude", "Altitude", "Accuracy"]], ignore_index=True)

def export_schema(df, form_cols):
    schema = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values,
                           "missing": df.isna().sum().values, "distinct": df.nunique().values})
    schema = schema.merge(form_cols, on="column", how="left")
    schema["form_type"] = schema["form_type"].fillna("ODK system column")
    return schema


survey_form = data_dir / "ODK Data Collection Forms" / "herbaceous_veg_survey.xlsx"

quadrat_schema = export_schema(
    herbaceous_veg_survey_quadrat_repeat,
    form_columns(survey_form, "quadrat_repeat"),
)
quadrat_schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,quadrat_number,int64,0,20,calculate,Quadrat Number
1,note_new_quadrat,float64,640,0,note,Move to **Quadrat ${quadrat_number}**
2,herbs_present,str,0,2,select_one yes_no,Are there any herbaceous species present?
3,herb_species-all_unknown_unidentifiable,str,7,2,select_one yes_no,Are **ALL** of the specimens either unidentifi...
4,herb_species-selected_herb_species_uuids,str,146,225,select_multiple species.csv,Select **ALL** of the species present:
5,herb_species-count_herb_species,float64,7,7,calculate,calculated: count-selected(${selected_herb_spe...
6,herb_species-note_herb_species_count,float64,640,0,note,**${count_herb_species}** species selected\n
7,additional_species_present,str,7,2,select_one yes_no,Are there any species present in the quadrat t...
8,species_list-known_species_list,str,146,188,calculate,calculated: if(${selected_herb_species_uuids} ...
9,species_list-additional_species_list,str,66,409,calculate,"calculated: join('<br/>', ${validated_name}[. ..."


### Columns kept for the checks: quadrat_repeat

`herbaceous_veg_survey-quadrat_repeat` has 21 columns. I kept 14: the ones needed to check that each survey has its 20 quadrats, that species and photos were recorded, and that locations are accurate.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `PARENT_KEY` | Identify each quadrat and link it to its survey |
| `quadrat_number` | Each survey should have quadrats 1 to 20, once each |
| `herbs_present`, `herb_species-all_unknown_unidentifiable` | Find quadrats with no species recorded, or where nothing could be matched to the species list |
| `herb_species-selected_herb_species_uuids`, `herb_species-count_herb_species` | The species recorded, and a cross-check that the count matches the number selected |
| `additional_species_present`, `species_list-all_species_list` | Link to the extra-species file, and a readable combined species list |
| `quadrat_image` | Check that a photo was taken for each quadrat |
| Latitude, longitude, accuracy | Check that the location is present, accurate (5 m or better per the SOP), and on the transect |
| `quadrat_comment` | Notes on unusual conditions (only a few are filled in) |

**What I dropped**

- **Empty notes.** `note_new_quadrat`, `herb_species-note_herb_species_count`, `species_list-note_species_summary_heading` and `species_list-note_species_list` are blank in all 640 rows.
- **Display-only lists.** `species_list-known_species_list` and `species_list-additional_species_list` are text the form builds from other columns, so they can be recreated from the species selections and the extra-species file.
- **Altitude.** The SOP sets no rule for altitude, so `location_quadrat-Altitude` supports no check.

In [89]:
quadrat_keep_cols = [
    "KEY", "PARENT_KEY", "quadrat_number",
    "herbs_present", "herb_species-all_unknown_unidentifiable",
    "herb_species-selected_herb_species_uuids", "herb_species-count_herb_species",
    "additional_species_present", "species_list-all_species_list",
    "quadrat_image",
    "location_quadrat-Latitude", "location_quadrat-Longitude", "location_quadrat-Accuracy",
    "quadrat_comment",
]

herbaceous_veg_survey_quadrat_repeat_keep = herbaceous_veg_survey_quadrat_repeat[quadrat_keep_cols]

herbaceous_veg_survey_quadrat_repeat_keep.shape   # should be (640, 14)

(640, 14)

### herbaceous_veg_survey-additional_species_repeat

This file holds the additional species records. Each row is one species recorded in a quadrat that was not on the approved species list, could not be identified in the field, or was an unknown or missing species already recorded earlier in the same submission. The 1,296 rows come from 574 of the 640 quadrats; the other 66 quadrats had no additional species. The `species_entry_mode` column shows how each species was recorded: as a new unknown (`new_unknown`), a new missing species typed in by name (`new_missing`), or a reused unknown or missing species (`reuse_unknown`, `reuse_missing`). It links to its quadrat through `PARENT_KEY` (matching the quadrat's `KEY`), and the reused species link to the `species_extra` entity list. New records are written to `species_extra` with a pending review status, so Natural State can confirm the identification. These records matter for richness because unknown and missing species are recorded under provisional names, and how to count them needs to be decided.

In [90]:
herbaceous_veg_survey_additional_species_repeat_schema = export_schema(
    herbaceous_veg_survey_additional_species_repeat,
    form_columns(survey_form, "additional_species_repeat"),
)
herbaceous_veg_survey_additional_species_repeat_schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,observation_index,int64,0,6,calculate,calculated: position(..)
1,species_entry_mode,str,0,4,select_one species_entry_mode,How do you want to record this species?
2,count_prior_new_in_submission,float64,1189,16,calculate,calculated: count(/data/observations/quadrat_r...
3,target_extra_name,str,1189,104,calculate,calculated: instance('species_extra')/root/ite...
4,assigned_unknown_index,float64,1189,104,calculate,calculated: int(${base_active_herb_count}) + i...
5,auto_new_unknown_label,str,1234,60,calculate,calculated: instance('species_extra')/root/ite...
6,note_new_unknown,float64,1296,0,note,New Record: **${auto_new_unknown_label}**\n\nT...
7,select_reuse_unknown,str,685,44,select_one_from_file species_extra.csv,Select the **unknown** to reuse
8,reused_unknown_label,str,685,44,calculate,calculated: instance('species_extra')/root/ite...
9,note_reuse_unknown,float64,1296,0,note,Linked to **${reused_unknown_label}**


### Columns kept for the checks: additional_species_repeat

`herbaceous_veg_survey-additional_species_repeat` has 30 columns. I kept 12: the ones needed to check that each additional species is linked to a quadrat, was recorded with a valid name, and has the photo and review status the form requires.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `PARENT_KEY` | Identify each record and link it to its quadrat |
| `observation_index` | Position of the record within its quadrat |
| `species_entry_mode` | How the species was recorded, which decides which other columns should be filled |
| `select_reuse_unknown`, `select_reuse_missing` | Link to `species_extra`, to check each reused species exists |
| `new_missing_canonical` | The species name typed in the field, to check its format (`Genus_species`) and spelling |
| `validated_name` | The final species name for the record, used to count species. It should never be empty |
| `entity_label_to_write`, `review_status`, `new_record_reason` | The new records written to `species_extra`, and whether they are still awaiting review |
| `macro_images-macro_1` | The close-up photo, which the form requires for every new unknown species |

**What I dropped**

- **Empty notes.** `note_new_unknown`, `note_reuse_unknown`, `note_new_missing`, `note_reuse_missing` and `macro_images-note_images` are blank in all 1,296 rows.
- **Calculation helpers.** `count_prior_new_in_submission`, `assigned_unknown_index` and `target_extra_name` are counters the form uses to assign new records. `auto_new_unknown_label`, `reused_unknown_label`, `auto_new_missing_label` and `reused_missing_label` are display labels that can be recreated from `species_extra`.
- **Constant or duplicate columns.** `write_choice_status` is always `active`, and `meta-entity-label` is identical to `entity_label_to_write`.
- **Optional extra photos.** `macro_images-macro_2` to `macro_images-macro_5` are optional, and three of the four are empty.

**One thing I noticed while choosing, to check later**

- `validated_name` is empty in 1 of the 1,296 rows.

In [91]:
additional_species_repeat_keep_cols = [
    "KEY", "PARENT_KEY", "observation_index",
    "species_entry_mode", "select_reuse_unknown", "select_reuse_missing",
    "new_missing_canonical", "validated_name",
    "entity_label_to_write", "review_status", "new_record_reason",
    "macro_images-macro_1",
]

herbaceous_veg_survey_additional_species_repeat_keep = herbaceous_veg_survey_additional_species_repeat[additional_species_repeat_keep_cols]

herbaceous_veg_survey_additional_species_repeat_keep.shape   # should be (1296, 12)

(1296, 12)

### register_vegetation_plots

This file holds the plot registrations. Each row is one registration of a vegetation plot (a 50 m × 5 m belt transect) by a field team, using the plot registration form. The 31 rows are 31 distinct plots. The team picks a planned sampling point from the `centroids` list and decides whether the plot is viable. If it is, they record its ecosystem type, woody vegetation, soil, ground cover and disturbance, plus photos and GPS locations at endpoint A, the midpoint and endpoint B. Registration creates the plot's row in the `vegplots` entity list, which the survey form's plot picker then draws from, and updates the matching centroid. In this export 30 plots are viable and 1 is not, and the non-viable plot has no metadata or locations. It links to `centroids` through `plot_selection-selected_plot_uuid`, to `surveys` through `survey_begin-selected_survey_uuid_multi`, and to `ecosystem_types` through `plot_metadata-selected_ecosystem_type_uuid`.

In [92]:
register_form = data_dir / "ODK Data Collection Forms" / "register_vegetation_plots.xlsx"

register_vegetation_plots_schema = export_schema(
    register_vegetation_plots,
    form_columns(register_form),
)
register_vegetation_plots_schema

/Users/scohen/Documents/GitHub/NS_SDS_TakeHomeProject_SCohen/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,column,dtype,missing,distinct,form_type,description
0,SubmissionDate,str,0,31,ODK system column,NaN
1,note_overview,float64,31,0,note,This form is to be used by field data teams to...
2,field_team_specifics-selected_transport_type_uuid,str,0,1,select_one transport.csv,Select the the mode of transport:
3,field_team_specifics-transport_type_label,str,0,1,calculate,"calculated: instance(""transport"")/root/item[na..."
4,field_team_specifics-selected_project_team_uui...,str,0,1,select_multiple_from_file project_team.csv,Select all the team members involved in this c...
...,...,...,...,...,...,...
77,Status,float64,31,0,ODK system column,NaN
78,ReviewState,float64,31,0,ODK system column,NaN
79,DeviceID,str,0,2,ODK system column,NaN
80,Edits,int64,0,1,ODK system column,NaN


### Columns kept for the checks: register_vegetation_plots

`register_vegetation_plots` has 82 columns. I kept 28: the ones needed to check who registered which plot and when, that the plot comes from a planned sampling point, and that the transect locations are present and consistent with the SOP.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `KEY`, `SubmissionDate`, start and end time | Identify each registration, and check dates and duration |
| Team (recorder and team members) | Confirm the recorder is a valid team member |
| `survey_begin-selected_survey_uuid_multi` | Which surveys the plot is assigned to |
| `selected_plot_uuid`, `plot_name`, `geohash`, `sample_status` | Plot comes from a planned centroid, is primary or backup, and its midpoint matches the geohash |
| `is_plot_viable`, `plot_establishment_notes` | A non-viable plot must have a reason |
| `selected_ecosystem_type_uuid`, `disturbance_types` | Valid ecosystem type, and disturbance such as fire that makes a plot non-viable |
| Latitude, longitude and accuracy at endpoint A, the midpoint and endpoint B | Transect length (50 m), orientation, midpoint within 25 m of the planned point, and accuracy of 5 m or better |
| `AttachmentsPresent`, `AttachmentsExpected`, `confirm_submission` | Missing photos, and confirmed submission |

**What I dropped**

- **Empty notes.** `note_overview`, `survey_begin-note_parent_labels`, `transect_orientation_note`, `survey_end-note_field_submission` and `survey_end-plot_establishment_notes` are blank in all 31 rows.
- **Columns that never vary or are copied from other lists.** Transport, default surveys, survey year, survey label, project, area, baseline year, replicate, stratum and plot label are the same in every row or are looked up from the `surveys` and `centroids` lists.
- **Habitat description.** The woody species, canopy, soil, ground cover and water questions describe the habitat, but the SOP sets no rule for them and this task is about herbaceous vegetation. `ecosystem_type_name` is a lookup of the ecosystem id I kept.
- **Photos, altitude and joined locations.** The four individual photo columns are replaced by the attachment counts. The SOP sets no rule for altitude. `vegplot_transect` is the three locations joined into one text. `background_geopoint` is empty in 16 of the 31 rows and is not needed because the transect points are recorded separately.
- **ODK internals.** `update_vegplot_link`, `update_marker_color`, the `meta-*` columns, `SubmitterID`, `SubmitterName`, `Status`, `ReviewState`, `DeviceID`, `Edits` and `FormVersion` are system columns, and `Status` and `ReviewState` are empty.

**Things I noticed while choosing, to check later**

- 30 plots are viable and 1 is not. The non-viable plot has no ecosystem, metadata or locations.
- 30 plots are primary sampling points and 1 is a backup.

In [93]:
register_vegetation_plots_keep_cols = [
    "KEY", "SubmissionDate", "survey_begin-start_time", "survey_end-end_time",
    "field_team_specifics-recorder_uuid", "field_team_specifics-recorder_label",
    "field_team_specifics-selected_project_team_uuid_multi",
    "survey_begin-selected_survey_uuid_multi",
    "plot_selection-selected_plot_uuid", "plot_selection-plot_name",
    "plot_selection-geohash", "plot_selection-sample_status",
    "plot_selection-is_plot_viable", "plot_selection-plot_establishment_notes",
    "plot_metadata-selected_ecosystem_type_uuid", "plot_metadata-disturbance_types",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Latitude",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Longitude",
    "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Accuracy",
    "plot_metadata-collect_points-midpoint-location_midpoint-Latitude",
    "plot_metadata-collect_points-midpoint-location_midpoint-Longitude",
    "plot_metadata-collect_points-midpoint-location_midpoint-Accuracy",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Latitude",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Longitude",
    "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Accuracy",
    "AttachmentsPresent", "AttachmentsExpected", "survey_end-confirm_submission",
]

register_vegetation_plots_keep = register_vegetation_plots[register_vegetation_plots_keep_cols]

register_vegetation_plots_keep.shape   # should be (31, 28)

(31, 28)

### vegplots

This is an entity list, a reference table rather than field data. Each row is one registered vegetation plot, created when the field team submits the plot registration form. The 31 rows match the 31 registrations. The survey form's plot picker draws from this list and offers only plots that belong to the selected survey and are marked viable. Each row records the plot's name, the geohash of its midpoint, whether it is a primary or backup sampling point, whether it is viable, the midpoint location (`geometry`: latitude, longitude, altitude and accuracy), and the two surveys it belongs to. The survey file's `plot_selection-selected_plot_uuid` matches `__id` here, and `plot_uuid` matches `__id` in the `centroids` list, the planned sampling point. The columns that start with `__` are added by ODK Central.

In [94]:
def table_profile(df):
    return pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values,
                         "missing": df.isna().sum().values, "distinct": df.nunique().values,
                         "unique": [df[c].is_unique and df[c].notna().all() for c in df.columns]})


vegplots_profile = table_profile(vegplots)
vegplots_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,31,True
1,label,str,0,31,True
2,replicate,str,0,1,False
3,plot_uuid,str,0,31,True
4,plot_name,str,0,31,True
5,geohash,str,0,31,True
6,plot_status,str,0,2,False
7,geometry,str,1,30,False
8,survey_labels,str,0,1,False
9,area_label,str,0,1,False


### Columns kept for the checks: vegplots

`vegplots` has 19 columns. I kept 10: the ones needed to check that surveyed plots are registered, viable and assigned to the right survey, and to compare each plot's midpoint with its planned location.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the survey file uses to pick a plot |
| `plot_uuid` | Link to the planned sampling point in `centroids` |
| `plot_name`, `geohash` | Identify the plot, and check its midpoint against the geohash cell |
| `plot_status` | Primary or backup sampling point |
| `is_viable` | The survey form offers only viable plots, so a survey of a non-viable plot is a problem |
| `geometry` | The midpoint location (latitude, longitude, altitude and accuracy) |
| `survey_uuids` | The surveys the plot is assigned to, to check a survey matches its plot |
| `__createdAt`, `__creatorName` | When the plot was registered and by which team |

**What I dropped**

- **Columns that never vary.** `replicate`, `survey_labels`, `area_label` and `stratum_label` have a single value in all 31 rows, and `__updates` and `__version` do too.
- **Duplicates and empty columns.** `label` is identical to `plot_name`, and `__updatedAt` is empty in every row.
- **ODK internals.** `__creatorId` is an id for the team already shown by `__creatorName`.

**Things I noticed while choosing, to check later**

- 30 plots are viable and 1 is not. The non-viable plot is a primary plot, and it is the only row with no `geometry`.
- 30 plots are primary and 1 is a backup, and the backup is a different plot from the non-viable one.
- Every plot belongs to 2 surveys.

In [95]:
vegplots_keep_cols = [
    "__id", "plot_uuid", "plot_name", "geohash", "plot_status",
    "is_viable", "geometry", "survey_uuids", "__createdAt", "__creatorName",
]

vegplots_keep = vegplots[vegplots_keep_cols]

vegplots_keep.shape   # should be (31, 10)

(31, 10)

### centroids

This is an entity list of the planned sampling points. Each row is one centroid: the location a field team navigates to and uses as the midpoint of a plot. There are 51 centroids, 36 primary and 15 backup, in two strata (savanna and shrubland). During plot registration the team picks a centroid from this list. The registration then creates a row in `vegplots` and marks the centroid as linked (`vegplot_linked` is `yes`). 31 centroids are linked, matching the 31 registered plots, and 20 have not been registered. The `geometry` column holds the planned location as four space-separated values: latitude, longitude, altitude and accuracy. All locations use the coordinate system EPSG:4326 (latitude and longitude). `choice_name` is the plot name, and `label` is that name with the status added, such as `(backup)`. `vegplots.plot_uuid` matches `__id` here. The columns that start with `__` are added by ODK Central.

In [96]:
centroids_profile = table_profile(centroids)
centroids_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,51,True
1,label,str,0,51,True
2,choice_name,str,0,51,True
3,project_label,str,0,1,False
4,project_abbr,str,0,1,False
5,project_area_name,str,0,1,False
6,crs,str,0,1,False
7,geohash,str,0,51,True
8,sample_status,str,0,2,False
9,geometry,str,0,51,True


### Columns kept for the checks: centroids

`centroids` has 21 columns. I kept 7: the ones needed to compare each registered plot with its planned location, and to find planned points that have not been registered.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key that `vegplots.plot_uuid` and the registration form point to |
| `choice_name` | The plot name, to match `vegplots` and the survey data |
| `geohash`, `geometry` | The planned location of the plot's midpoint, to compare with the registered midpoint (it should be within 25 m) |
| `sample_status` | Primary or backup sampling point |
| `stratum_label` | The habitat stratum the point was drawn from |
| `vegplot_linked` | Whether the point has been registered, to find planned points with no plot |

**What I dropped**

- **Columns that never vary.** `project_label`, `project_abbr`, `project_area_name`, `crs` and `replicate` have a single value in all 51 rows. The coordinate system is EPSG:4326 for every row.
- **Duplicates and display columns.** `label` is the plot name plus its status, which are already in `choice_name` and `sample_status`. `marker-color` is the colour the point shows on the form's map.
- **Mostly empty.** `survey_year` is empty for the 19 points that were never registered.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Things I noticed while choosing, to check later**

- 31 of the 51 planned points are registered and 20 are not (6 primary and 14 backup).
- `vegplot_linked` is `yes` for exactly the 31 centroids that appear in `vegplots`.

In [97]:
centroids_keep_cols = [
    "__id", "choice_name", "geohash", "geometry",
    "sample_status", "stratum_label", "vegplot_linked",
]

centroids_keep = centroids[centroids_keep_cols]

centroids_keep.shape   # should be (51, 7)

(51, 7)

### species

This is an entity list of the approved species for the project. Each row is one plant species, 370 in total: 238 herbaceous and 132 woody. The `groups` column says which, and the survey form offers only the `herbaceous` species in its species picker. The survey file's species selections (`herb_species-selected_herb_species_uuids`) match `__id` here. `label` is the species name without authorship, which is the name shown on the form, and `scientific_name` adds the authority. The other columns give the taxonomy (family, genus, and so on) and where the name came from. The columns that start with `__` are added by ODK Central.

In [98]:
species_profile = table_profile(species)
species_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,370,True
1,label,str,0,370,True
2,scientific_name,str,0,370,True
3,vernacular_name,float64,370,0,False
4,rank,str,0,1,False
5,kingdom,str,0,1,False
6,phylum,str,0,1,False
7,class,str,7,2,False
8,order,str,7,32,False
9,family,str,0,68,False


### Columns kept for the checks: species

`species` has 21 columns. I kept 6: the ones needed to check that recorded species are on the approved list and are herbaceous, and to summarize species by family and genus.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the quadrat file uses for each selected species |
| `label`, `scientific_name` | The species name, to report which species were recorded |
| `groups` | Herbaceous or woody, because only herbaceous species should be recorded here |
| `family`, `genus` | To summarize richness and diversity by family or genus |

**What I dropped**

- **Empty or constant columns.** `vernacular_name` and `global_taxon_id` are empty in all 370 rows, and `rank`, `kingdom` and `phylum` have a single value.
- **Detail not needed for the checks.** `class`, `order`, `sources` and `gbif_taxon_key` describe the taxonomy and where each name came from.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Thing I noticed while choosing, to check later**

- The approved list has 132 woody species. A woody species recorded in a herbaceous quadrat would be a problem.

In [99]:
species_keep_cols = ["__id", "label", "scientific_name", "family", "genus", "groups"]

species_keep = species[species_keep_cols]

species_keep.shape   # should be (370, 6)

(370, 6)

### species_extra

This is an entity list of the species added in the field. Each row is one record the form can use for an unknown species (one that could not be identified) or a missing species (one not on the approved list). Of the 250 rows, 200 are herbaceous and 50 are woody. Many rows are inactive (141): the form holds them in reserve and activates the next one when a new species is recorded, which also sets its review status to `pending` and gives the reason (`unknown_species` or `missing_species`). The additional species file's `select_reuse_unknown` and `select_reuse_missing` match `__id` here. The columns that start with `__` are added by ODK Central.

In [100]:
species_extra_profile = table_profile(species_extra)
species_extra_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,250,True
1,label,str,0,249,False
2,choice_status,str,0,2,False
3,unknown_index,int64,0,200,False
4,review_status,str,141,1,False
5,new_record_reason,str,141,2,False
6,group_type,str,0,2,False
7,new_assigned_uuid,float64,250,0,False
8,__createdAt,str,0,14,False
9,__creatorId,int64,0,1,False


### Columns kept for the checks: species_extra

`species_extra` has 14 columns. I kept 7: the ones needed to check that new and reused species exist, are active, and are waiting for review.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the additional species file uses for reused species |
| `label` | The species name, or the placeholder name for an unknown |
| `choice_status` | Active or inactive, to check that records in use are active |
| `unknown_index` | The number the form assigns to each new record |
| `review_status`, `new_record_reason` | Whether a new record is waiting for review, and why it was added |
| `group_type` | Herbaceous or woody |

**What I dropped**

- **Empty column.** `new_assigned_uuid` is empty in all 250 rows. It would hold the identified species once a record is resolved.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Things I noticed while choosing, to check later**

- `label` has 249 distinct values in 250 rows, so one label appears twice.
- 109 records are active and all of them are pending review. 104 of the active records are herbaceous.
- The additional species file has 107 new records (62 new unknown and 45 new missing), compared with 104 active herbaceous records here, so these are worth reconciling.

In [101]:
species_extra_keep_cols = [
    "__id", "label", "choice_status", "unknown_index",
    "review_status", "new_record_reason", "group_type",
]

species_extra_keep = species_extra[species_extra_keep_cols]

species_extra_keep.shape   # should be (250, 7)

(250, 7)

### project_team

This is an entity list of the people who can be chosen as team members or recorder on the forms. Each row is one person, 6 in total, all active. Five belong to Acacia Trust and one to Natural State. The survey and registration files' recorder and team member columns match `__id` here. The columns that start with `__` are added by ODK Central.

In [102]:
project_team_profile = table_profile(project_team)
project_team_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,6,True
1,label,str,0,6,True
2,choice_name,str,0,6,True
3,choice_status,str,0,1,False
4,first_name,str,0,6,True
5,family_name,str,0,6,True
6,affiliation,str,0,2,False
7,__createdAt,str,0,3,False
8,__creatorId,int64,0,2,False
9,__creatorName,str,0,2,False


### Columns kept for the checks: project_team

`project_team` has 13 columns. I kept 5: the ones needed to check that the recorder and team members on each submission are valid and active.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the survey and registration files use for the recorder and team members |
| `label`, `choice_name` | The name shown on the form, and the person's full name |
| `choice_status` | Whether the person is active, because the form offers only active people |
| `affiliation` | The organization, to summarize who did the work |

**What I dropped**

- **Name parts.** `first_name` and `family_name` are already combined in `choice_name`.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

In [103]:
project_team_keep_cols = ["__id", "label", "choice_name", "choice_status", "affiliation"]

project_team_keep = project_team[project_team_keep_cols]

project_team_keep.shape   # should be (6, 5)

(6, 5)

### surveys

This is an entity list of the surveys defined for the project. Each row is one survey: a combination of project, area, survey type and target group. There are 4: woody and herbaceous vegetation (human observation), mammals (camera trapping) and birds (passive acoustic monitoring), all in the Lewa area with a 2026 baseline. The survey file's `survey_begin-selected_survey_uuid` and the registration file's `survey_begin-selected_survey_uuid_multi` match `__id` here. The herbaceous survey form offers only the survey whose `target_group` is `herbs`, and `vegplots.survey_uuids` lists the surveys each plot belongs to. The columns that start with `__` are added by ODK Central.

In [104]:
surveys_profile = table_profile(surveys)
surveys_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,4,True
1,label,str,0,4,True
2,project_abbr,str,0,1,False
3,survey_type_label,str,0,3,False
4,area_label,str,0,1,False
5,target_group,str,0,4,True
6,baseline_year,int64,0,1,False
7,project_label,str,0,1,False
8,target_parent_taxon,str,0,2,False
9,__createdAt,str,0,4,True


### Columns kept for the checks: surveys

`surveys` has 15 columns. I kept 4: the ones needed to check that each submission and plot points to the right survey.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the survey, registration and `vegplots` files use |
| `label` | The survey name, such as `savmon\|lw\|herbs\|ho` |
| `survey_type_label` | Human observation, camera trapping or passive acoustic monitoring |
| `target_group` | Herbs, woody, mammals or birds, to check a herbaceous submission uses the `herbs` survey |

**What I dropped**

- **Columns that never vary.** `project_abbr`, `project_label`, `area_label` and `baseline_year` have a single value in all 4 rows. The baseline year is 2026.
- **Derived column.** `target_parent_taxon` (plants or vertebrates) follows from the target group.
- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns.

**Thing I noticed while choosing, to check later**

- Only 2 of the 4 surveys (woody and herbs) apply to vegetation plots, and every plot in `vegplots` belongs to 2 surveys.

In [105]:
surveys_keep_cols = ["__id", "label", "survey_type_label", "target_group"]

surveys_keep = surveys[surveys_keep_cols]

surveys_keep.shape   # should be (4, 4)

(4, 4)

### species_extra_ids

This is an entity list that links the placeholder names used for new species to species names, where they are known. Each row is one placeholder (`species_number`, such as `wood_003`) with the species name it was identified as. The 109 rows include 32 where the name is still `Unknown`, and 6 have a note. `species_extra` holds the records the form creates, and this list gives the identification of those placeholders as it is worked out. The columns that start with `__` are added by ODK Central.

In [106]:
species_extra_ids_profile = table_profile(species_extra_ids)
species_extra_ids_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,109,True
1,label,str,0,98,False
2,species_number,str,0,109,True
3,species_name,str,0,69,False
4,notes,str,103,4,False
5,__createdAt,str,0,35,False
6,__creatorId,int64,0,1,False
7,__creatorName,str,0,1,False
8,__updates,int64,0,1,False
9,__updatedAt,float64,109,0,False


### Columns kept for the checks: species_extra_ids

`species_extra_ids` has 11 columns. I kept 5: the ones needed to see which placeholders have been identified.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key for each row |
| `label` | The name shown for the placeholder |
| `species_number` | The placeholder id, such as `wood_003`, to link to `species_extra` |
| `species_name` | The species it was identified as, or `Unknown` |
| `notes` | Comments on the identification (6 rows have one) |

**What I dropped**

- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns, and `__updatedAt` is empty in every row.

**Things I noticed while choosing, to check later**

- `label` has 98 distinct values in 109 rows, so some labels repeat.
- 32 of the 109 rows are still `Unknown`.
- Only 64 of the 109 `species_number` values appear as a label in `species_extra`, so the link between the two lists is partial.

In [107]:
species_extra_ids_keep_cols = ["__id", "label", "species_number", "species_name", "notes"]

species_extra_ids_keep = species_extra_ids[species_extra_ids_keep_cols]

species_extra_ids_keep.shape   # should be (109, 5)

(109, 5)

### ecosystem_types

This is an entity list of the ecosystem types a plot can be assigned during registration. There are 5: grassland, forest, scrub, rocky outcrop and riverine area, all active. The registration file's `plot_metadata-selected_ecosystem_type_uuid` matches `__id` here. In the registration data, 29 plots are scrub and 1 is forest, and the non-viable plot has none. The columns that start with `__` are added by ODK Central.

In [108]:
ecosystem_types_profile = table_profile(ecosystem_types)
ecosystem_types_profile

,column,dtype,missing,distinct,unique
0,__id,str,0,5,True
1,label,str,0,5,True
2,choice_name,str,0,5,True
3,choice_status,str,0,1,False
4,__createdAt,str,0,2,False
5,__creatorId,int64,0,1,False
6,__creatorName,str,0,1,False
7,__updates,int64,0,1,False
8,__updatedAt,float64,5,0,False
9,__version,int64,0,1,False


### Columns kept for the checks: ecosystem_types

`ecosystem_types` has 10 columns. I kept 4: the ones needed to check that a registered plot's ecosystem is a valid, active type.

**What I kept, and the checks each group supports**

| Columns | Why it is kept |
|---|---|
| `__id` | The key the registration file uses |
| `label`, `choice_name` | The ecosystem name, and the name used by the form |
| `choice_status` | Whether the type is active, because the form offers only active types |

**What I dropped**

- **ODK internals.** `__createdAt`, `__creatorId`, `__creatorName`, `__updates`, `__updatedAt` and `__version` are system columns, and `__updatedAt` is empty in every row.

In [109]:
ecosystem_types_keep_cols = ["__id", "label", "choice_name", "choice_status"]

ecosystem_types_keep = ecosystem_types[ecosystem_types_keep_cols]

ecosystem_types_keep.shape   # should be (5, 4)

(5, 4)

## Planned checks

Each check flags problems and does not fix them. The rule names the columns involved, and the source says where the rule comes from. "Herbaceous SOP" and "Plot Registration SOP" are the two SOP PDFs, and "survey form" and "registration form" are the two ODK forms (`survey` sheet). Priority: M = must, S = should, O = optional if time allows.

In [31]:
planned_checks = pd.DataFrame(
    [
        (1, "Survey", "20 quadrats per survey",
         "For each survey `KEY`, the number of rows in `quadrat_repeat` with that `PARENT_KEY` is 20, and `observations-quadrat_repeat_count` is 20",
         "Herbaceous SOP section 3.1 (fixed count of 20 quadrats per plot) and section 5 step 13; survey form: repeat `quadrat_repeat` has `repeat_count` 20", "M"),
        (2, "Survey", "Quadrat numbers 1 to 20, each once",
         "For each `PARENT_KEY`, `quadrat_number` is exactly 1 to 20 with no gaps or repeats",
         "Herbaceous SOP section 5 steps 11 and 12 (quadrats every 5 m); survey form: `quadrat_number` is `position(..)` in `quadrat_repeat`", "M"),
        (3, "Survey", "Plot is registered, viable and in the herbs survey",
         "`plot_selection-selected_plot_uuid` is in `vegplots.__id`, that row has `is_viable` = yes, and its `survey_uuids` includes `survey_begin-selected_survey_uuid`",
         "Herbaceous SOP section 5 step 4; survey form: `selected_plot_uuid` choice filter `selected(survey_uuids, ${selected_survey_uuid}) and is_viable = 'yes'`", "M"),
        (4, "Survey", "Repeated plot visits",
         "The same `plot_selection-selected_plot_uuid` appears in more than one submission (32 submissions, 30 distinct plots)",
         "Herbaceous SOP section 5 step 16 (move on to the next registered plot); data", "M"),
        (5, "Survey", "Recorder and team are valid",
         "`field_team_specifics-recorder_uuid` is one of `selected_project_team_uuid_multi`, and all are in `project_team.__id` with `choice_status` = active",
         "Survey form: `recorder_uuid` choice filter `selected(${selected_project_team_uuid_multi}, name)`; `selected_project_team_uuid_multi` choice filter `choice_status = 'active'`", "S"),
        (6, "Survey", "Timing",
         "`survey_begin-start_time` is before `survey_end-end_time`, which is not after `SubmissionDate`. No time limit is checked, because the SOPs give none",
         "Survey form: `start_time` (type `start`) and `end_time` (type `end`); Herbaceous SOP section 6 and Plot Registration SOP section 6 (submit as soon as a stable internet connection is available, with no time limit)", "S"),
        (7, "Survey", "Missing photos",
         "`AttachmentsPresent` is less than `AttachmentsExpected`",
         "ODK Central export columns; survey form: `quadrat_image` is required", "O"),
        (8, "Quadrat", "GPS accuracy",
         "`location_quadrat-Accuracy` is 5 or less (metres)",
         "Herbaceous SOP section 4.2 (accuracy of 5 m or better); survey form: `location_quadrat` has `capture-accuracy=5 warning-accuracy=10`", "M"),
        (9, "Quadrat", "Location is on the transect",
         "The quadrat (`location_quadrat-Latitude` and `-Longitude`) is 30 m or less from the plot midpoint (`vegplots.geometry`), no more than 12 m from the line between endpoints A and B, and no more than 5 m past either end. These limits are derived, not stated in the SOPs (see \"Check 9\" below)",
         "Plot Registration SOP section 2 (belt transect is 50 m by 5 m) and the Figure 1 caption in section 3 (endpoints 25 m from the midpoint); Herbaceous SOP section 4.2 (5 m accuracy) and section 5 steps 5 and 12", "S"),
        (10, "Quadrat", "Species logic",
         "`herbs_present` = yes means a species is recorded, and no means none; `herb_species-count_herb_species` equals the number of `selected_herb_species_uuids`; `all_unknown_unidentifiable` = yes means `additional_species_present` = yes",
         "Herbaceous SOP section 5 steps 6, 7 and 13; survey form: `additional_species_present` constraint `. = 'yes' or ${all_unknown_unidentifiable} != 'yes'`", "M"),
        (11, "Quadrat", "Species is valid and herbaceous",
         "Each uuid in `herb_species-selected_herb_species_uuids` is in `species.__id` with `groups` = herbaceous",
         "Herbaceous SOP section 5 step 7; survey form: `selected_herb_species_uuids` choice filter `groups = 'herbaceous'`", "S"),
        (12, "Quadrat", "Plots with no species",
         "Plots where `survey_end-quadrats_with_species` is 0",
         "Herbaceous SOP section 5 step 13 (0 herbaceous quadrats is implausible); survey form: `quadrats_with_species` calculation", "M"),
        (13, "Extra species", "Name is present",
         "`validated_name` is not empty (1 row is empty)",
         "Survey form: `validated_name` calculation in `additional_species_repeat`; data", "M"),
        (14, "Extra species", "Name format",
         "`new_missing_canonical` is `Genus_species`: a capitalized genus and a lowercase epithet joined by an underscore, with no leading or trailing space (several names are separated by a single space)",
         "Herbaceous SOP section 2 (Canonical) and section 5 step 7; survey form: `new_missing_canonical` constraint only blocks `, ; ( )`, and its hint shows a space", "M"),
        (15, "Extra species", "A 'missing' species is already on the list",
         "`new_missing_canonical` (underscores read as spaces) matches `species.label`",
         "Herbaceous SOP section 5 step 7 (free text only for species not on the list); `species` entity list", "S"),
        (16, "Extra species", "Photos, and reused records exist and are active",
         "`macro_images-macro_1` is present when `species_entry_mode` = new_unknown; `select_reuse_unknown` and `select_reuse_missing` are in `species_extra.__id` with `choice_status` = active",
         "Survey form: `macro_1` is required for new_unknown; `select_reuse_unknown` choice filter on active records", "S"),
        (17, "Plot", "Midpoint is near the planned point",
         "Distance between the registered midpoint (`location_midpoint-Latitude` and `-Longitude`) and the plot's `centroids.geometry` is 25 m or less",
         "Plot Registration SOP section 5 step 5 (the midpoint cannot be moved more than 25 m)", "M"),
        (18, "Plot", "Transect length",
         "The distance from endpoint A to endpoint B is within the two recorded endpoint `-Accuracy` values added together of 50 m, and the midpoint is within its two `-Accuracy` values added together of 25 m from each endpoint. The tolerance is derived, not stated in the SOPs (see \"Check 18\" below)",
         "Plot Registration SOP section 2 (belt transect is 50 m long) and the Figure 1 caption in section 3 (endpoints 25 m from the midpoint); section 5 steps 6 and 7", "M"),
        (19, "Plot", "Registration GPS accuracy",
         "`location_endpoint_a`, `location_midpoint` and `location_endpoint_b` `-Accuracy` are 5 or less (metres)",
         "Registration form: location questions have `capture-accuracy=5 warning-accuracy=5`; Herbaceous SOP section 4.2", "S"),
        (20, "Plot", "Non-viable plots",
         "`plot_selection-is_plot_viable` = no has `plot_establishment_notes` filled, and the plot has no survey",
         "Plot Registration SOP section 5 step 3 (a non-viable plot must be clearly justified); registration form: `plot_establishment_notes` is required if not viable", "M"),
        (21, "Plot", "Fire on a surveyed plot",
         "`plot_metadata-disturbance_types` includes fire for a plot that was surveyed",
         "Plot Registration SOP section 5 step 3 (fire-damaged plots should not be sampled)", "O"),
        (22, "Plot", "Planned, registered and surveyed",
         "Compare the counts in `centroids` (51), `vegplots` (31, `centroids.vegplot_linked` = yes) and the survey file (30 distinct plots)",
         "`centroids` and `vegplots` entity lists; survey file", "M"),
        (23, "Quadrat", "Identical coordinates",
         "Two quadrats of the same survey have exactly the same `location_quadrat-Latitude` and `-Longitude`",
         "Herbaceous SOP section 5 steps 5, 9 and 11 (the point is recorded standing over each quadrat, and quadrats are never in the same place); data", "O"),
    ],
    columns=["id", "level", "check", "rule", "source", "priority"],
)

(
    planned_checks.style
    .hide(axis="index")
    .set_properties(**{"white-space": "normal", "text-align": "left", "vertical-align": "top"})
    .set_properties(subset=["rule", "source"], **{"min-width": "320px"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

id,level,check,rule,source,priority
1,Survey,20 quadrats per survey,"For each survey `KEY`, the number of rows in `quadrat_repeat` with that `PARENT_KEY` is 20, and `observations-quadrat_repeat_count` is 20",Herbaceous SOP section 3.1 (fixed count of 20 quadrats per plot) and section 5 step 13; survey form: repeat `quadrat_repeat` has `repeat_count` 20,M
2,Survey,"Quadrat numbers 1 to 20, each once","For each `PARENT_KEY`, `quadrat_number` is exactly 1 to 20 with no gaps or repeats",Herbaceous SOP section 5 steps 11 and 12 (quadrats every 5 m); survey form: `quadrat_number` is `position(..)` in `quadrat_repeat`,M
3,Survey,"Plot is registered, viable and in the herbs survey","`plot_selection-selected_plot_uuid` is in `vegplots.__id`, that row has `is_viable` = yes, and its `survey_uuids` includes `survey_begin-selected_survey_uuid`","Herbaceous SOP section 5 step 4; survey form: `selected_plot_uuid` choice filter `selected(survey_uuids, ${selected_survey_uuid}) and is_viable = 'yes'`",M
4,Survey,Repeated plot visits,"The same `plot_selection-selected_plot_uuid` appears in more than one submission (32 submissions, 30 distinct plots)",Herbaceous SOP section 5 step 16 (move on to the next registered plot); data,M
5,Survey,Recorder and team are valid,"`field_team_specifics-recorder_uuid` is one of `selected_project_team_uuid_multi`, and all are in `project_team.__id` with `choice_status` = active","Survey form: `recorder_uuid` choice filter `selected(${selected_project_team_uuid_multi}, name)`; `selected_project_team_uuid_multi` choice filter `choice_status = 'active'`",S
6,Survey,Timing,"`survey_begin-start_time` is before `survey_end-end_time`, which is not after `SubmissionDate`. No time limit is checked, because the SOPs give none","Survey form: `start_time` (type `start`) and `end_time` (type `end`); Herbaceous SOP section 6 and Plot Registration SOP section 6 (submit as soon as a stable internet connection is available, with no time limit)",S
7,Survey,Missing photos,`AttachmentsPresent` is less than `AttachmentsExpected`,ODK Central export columns; survey form: `quadrat_image` is required,O
8,Quadrat,GPS accuracy,`location_quadrat-Accuracy` is 5 or less (metres),Herbaceous SOP section 4.2 (accuracy of 5 m or better); survey form: `location_quadrat` has `capture-accuracy=5 warning-accuracy=10`,M
9,Quadrat,Location is on the transect,"The quadrat (`location_quadrat-Latitude` and `-Longitude`) is 30 m or less from the plot midpoint (`vegplots.geometry`), no more than 12 m from the line between endpoints A and B, and no more than 5 m past either end. These limits are derived, not stated in the SOPs (see ""Check 9"" below)",Plot Registration SOP section 2 (belt transect is 50 m by 5 m) and the Figure 1 caption in section 3 (endpoints 25 m from the midpoint); Herbaceous SOP section 4.2 (5 m accuracy) and section 5 steps 5 and 12,S
10,Quadrat,Species logic,"`herbs_present` = yes means a species is recorded, and no means none; `herb_species-count_herb_species` equals the number of `selected_herb_species_uuids`; `all_unknown_unidentifiable` = yes means `additional_species_present` = yes","Herbaceous SOP section 5 steps 6, 7 and 13; survey form: `additional_species_present` constraint `. = 'yes' or ${all_unknown_unidentifiable} != 'yes'`",M


## Running the checks

Each check adds one row per problem to `flags_table` (columns `check_id`, `table`, `key`, `detail`). Nothing is fixed or removed. The summary at the end counts the flags per check and does not list the rows, because the survey data are not to be published. Distances are computed in meters in UTM zone 37N (EPSG:32637).

Checks 9 and 18 use limits that I derived. The SOPs do not state them, and the reasoning is given before each check.

In [32]:
import geopandas as gpd
import numpy as np

# ---------- helpers ----------
flag_tables = []

def add_flags(check_id, table, flagged, key, detail=""):
    """Record one flag for each flagged row, identified by its key column."""
    flag_tables.append(pd.DataFrame({
        "check_id": check_id,
        "table": table,
        "key": flagged[key].values,
        "detail": detail if isinstance(detail, str) else detail.values,
    }))

def to_points(lat, lon):
    """Points in a metric coordinate system (UTM zone 37N) so distances are in metres."""
    return gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs("EPSG:32637")

survey = herbaceous_veg_survey_keep
quadrats = herbaceous_veg_survey_quadrat_repeat_keep
extra = herbaceous_veg_survey_additional_species_repeat_keep
register = register_vegetation_plots_keep

plot_col = "plot_selection-selected_plot_uuid"

### Survey checks (1 to 7)

Check 6 tests order only: the start is before the end, and the end is not after the submission. The SOPs give no time limit for a survey. They only say to submit saved forms as soon as a stable internet connection is available (Herbaceous SOP section 6 and Plot Registration SOP section 6), so no limit on duration or on submission lag is checked.

In [33]:
# ---------- 1. 20 quadrats per survey ----------
quadrat_counts = quadrats.groupby("PARENT_KEY").size().rename("n_quadrats")
s1 = survey.merge(quadrat_counts, left_on="KEY", right_index=True, how="left")
s1["n_quadrats"] = s1["n_quadrats"].fillna(0).astype(int)
bad = s1[(s1["n_quadrats"] != 20) | (s1["observations-quadrat_repeat_count"] != 20)]
add_flags(1, "herbaceous_veg_survey", bad, "KEY",
          "quadrat rows = " + bad["n_quadrats"].astype(str) + ", repeat count = " + bad["observations-quadrat_repeat_count"].astype(str))

# ---------- 2. quadrat numbers 1 to 20, each once ----------
numbers_ok = quadrats.groupby("PARENT_KEY")["quadrat_number"].apply(lambda x: sorted(x) == list(range(1, 21)))
bad = survey[survey["KEY"].isin(numbers_ok[~numbers_ok].index)]
add_flags(2, "herbaceous_veg_survey", bad, "KEY", "quadrat numbers are not exactly 1 to 20")

# ---------- 3. plot is registered, viable and in the herbs survey ----------
s3 = survey.merge(vegplots_keep[["__id", "is_viable", "survey_uuids"]],
                  left_on=plot_col, right_on="__id", how="left")
not_registered = s3["__id"].isna()
not_viable = s3["is_viable"].ne("yes") & ~not_registered
wrong_survey = ~not_registered & ~s3.apply(
    lambda r: isinstance(r["survey_uuids"], str) and r["survey_begin-selected_survey_uuid"] in r["survey_uuids"].split(), axis=1)
for mask, text in [(not_registered, "plot is not registered"), (not_viable, "plot is not viable"),
                   (wrong_survey, "plot does not belong to the survey selected")]:
    add_flags(3, "herbaceous_veg_survey", s3[mask], "KEY", text)

# ---------- 4. repeated plot visits ----------
visits = survey.groupby(plot_col)["KEY"].transform("size")
bad = survey[visits > 1]
add_flags(4, "herbaceous_veg_survey", bad, "KEY", "plot visited " + visits[visits > 1].astype(str) + " times")

# ---------- 5. recorder and team are valid ----------
team = survey["field_team_specifics-selected_project_team_uuid_multi"].str.split()
recorder = survey["field_team_specifics-recorder_uuid"]
active_people = set(project_team_keep.loc[project_team_keep["choice_status"] == "active", "__id"])
recorder_not_in_team = [r not in t for r, t in zip(recorder, team)]
person_not_active = [any(p not in active_people for p in t + [r]) for r, t in zip(recorder, team)]
add_flags(5, "herbaceous_veg_survey", survey[recorder_not_in_team], "KEY", "recorder is not one of the team members")
add_flags(5, "herbaceous_veg_survey", survey[person_not_active], "KEY", "recorder or team member is not an active person in project_team")

# ---------- 6. timing (order only) ----------
start = pd.to_datetime(survey["survey_begin-start_time"], utc=True)
end = pd.to_datetime(survey["survey_end-end_time"], utc=True)
submitted = pd.to_datetime(survey["SubmissionDate"], utc=True)
add_flags(6, "herbaceous_veg_survey", survey[end <= start], "KEY", "end time is not after start time")
add_flags(6, "herbaceous_veg_survey", survey[submitted < end], "KEY", "submitted before the survey ended")

# ---------- 7. missing photos ----------
bad = survey[survey["AttachmentsPresent"] < survey["AttachmentsExpected"]]
add_flags(7, "herbaceous_veg_survey", bad, "KEY",
          "attachments present " + bad["AttachmentsPresent"].astype(str) + " of " + bad["AttachmentsExpected"].astype(str))

### Registered transect points

Checks 9 and 17 to 21 use the points recorded at plot registration (endpoint A, midpoint, endpoint B). This cell joins them to the planned point and measures the distances between them in meters.

In [34]:
# Registered transect points, joined to the planned point, with distances in meters.
reg = register.merge(centroids_keep[["__id", "geometry"]], left_on="plot_selection-selected_plot_uuid", right_on="__id", how="left", suffixes=("", "_planned"))
a_lat, a_lon = "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Latitude", "plot_metadata-collect_points-endpoint_a-location_endpoint_a-Longitude"
m_lat, m_lon = "plot_metadata-collect_points-midpoint-location_midpoint-Latitude", "plot_metadata-collect_points-midpoint-location_midpoint-Longitude"
b_lat, b_lon = "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Latitude", "plot_metadata-collect_points-endpoint_b-location_endpoint_b-Longitude"
has_points = reg[m_lat].notna()
planned = reg["geometry"].str.split(expand=True).iloc[:, :2].astype(float)
pa, pm, pb = (to_points(reg[la], reg[lo]) for la, lo in [(a_lat, a_lon), (m_lat, m_lon), (b_lat, b_lon)])
pp = to_points(planned[0], planned[1])
reg = reg.assign(
    midpoint_to_planned_m=pm.distance(pp).values,
    a_to_b_m=pa.distance(pb).values,
    a_to_mid_m=pa.distance(pm).values,
    mid_to_b_m=pm.distance(pb).values,
)

### Check 9: where a quadrat can be

The SOPs do not say how far a quadrat can be from the plot midpoint or from the transect line. They give the geometry, and I derived the limits from it:

- The transect is 50 m long, and endpoints A and B are each 25 m from the midpoint (Plot Registration SOP, Figure 1 caption). The belt is 5 m wide (2.5 m each side of the line). Quadrats are placed every 5 m starting at endpoint A, with a right and a left quadrat at each mark. Each 1 m frame is set flush to the end of a 2.5 m stick, so it covers 1.5 to 2.5 m from the line and its center is about 2 m from the line (Herbaceous SOP section 5 steps 5, 11 and 12).
- Each GPS point has to be recorded with an accuracy of 5 m or better (Herbaceous SOP section 4.2).

From that, the farthest a correctly placed quadrat can be from the midpoint is about 25 m (the marks at 0 and 45 m), and GPS error can add up to 5 m, so the **midpoint limit is 30 m**.

The center of a quadrat should be about 2 m from the transect line. The line is itself drawn between two GPS points that can each be 5 m off, and the quadrat's own point can be 5 m off. Added to the 2 m, the **limit across the line is 12 m** (2 + 5 + 5). A quadrat can also sit slightly beyond an end of the line, so the **limit past either end is 5 m**, the GPS error of the quadrat. A limit of 5 m across the line would flag half of all quadrats, because it is the same size as the GPS error and cannot tell a misplaced quadrat from a noisy fix.

Both limits are derived, not rules in the SOPs. Quadrats whose plot has no registered points cannot be tested against the line and are counted in `n_quadrats_without_line`.

One observation, not a flag: the SOP steps 11 and 12 describe a right and a left quadrat at each 5 m mark, but in the data quadrats 1 to 10 run out from endpoint A along the line and 11 to 20 come back. This does not affect the limits above.

In [35]:
# ---------- 8. quadrat GPS accuracy (5 m or better) ----------
acc = quadrats["location_quadrat-Accuracy"]
bad = quadrats[(acc > 5) | acc.isna()]
add_flags(8, "quadrat_repeat", bad, "KEY", "accuracy " + bad["location_quadrat-Accuracy"].astype(str) + " m")

# ---------- 9. quadrat location ----------
q9 = (quadrats.merge(survey[["KEY", plot_col]], left_on="PARENT_KEY", right_on="KEY", suffixes=("", "_survey"))
              .merge(vegplots_keep[["__id", "plot_uuid", "geometry"]], left_on=plot_col, right_on="__id", how="left"))

# (a) distance from the plot midpoint (the registered midpoint in vegplots.geometry), limit 30 m
mid = q9["geometry"].str.split(expand=True).iloc[:, :2].astype(float)
pq = to_points(q9["location_quadrat-Latitude"], q9["location_quadrat-Longitude"])
q9["distance_m"] = pq.distance(to_points(mid[0], mid[1])).values
bad = q9[q9["distance_m"] > 30]
add_flags(9, "quadrat_repeat", bad, "KEY", "quadrat is " + bad["distance_m"].round(1).astype(str) + " m from the plot midpoint (limit 30 m)")

# (b) position relative to the A to B line: across the line (limit 12 m) and along it (limit 5 m past either end)
ends = reg.loc[has_points, ["plot_selection-selected_plot_uuid", a_lat, a_lon, b_lat, b_lon]].drop_duplicates("plot_selection-selected_plot_uuid")
q9 = q9.merge(ends, left_on="plot_uuid", right_on="plot_selection-selected_plot_uuid", how="left", suffixes=("", "_reg"))
n_quadrats_without_line = int(q9[a_lat].isna().sum())
q9 = q9[q9[a_lat].notna()].copy()
pq = to_points(q9["location_quadrat-Latitude"], q9["location_quadrat-Longitude"])
pa_q = to_points(q9[a_lat], q9[a_lon])
pb_q = to_points(q9[b_lat], q9[b_lon])
vx, vy = pb_q.x.values - pa_q.x.values, pb_q.y.values - pa_q.y.values
line_m = np.hypot(vx, vy)
dx, dy = pq.x.values - pa_q.x.values, pq.y.values - pa_q.y.values
q9["along_m"] = (dx * vx + dy * vy) / line_m
q9["across_m"] = np.abs(dx * vy - dy * vx) / line_m
q9["line_m"] = line_m
bad = q9[q9["across_m"] > 12]
add_flags(9, "quadrat_repeat", bad, "KEY", "quadrat is " + bad["across_m"].round(1).astype(str) + " m from the A to B line (limit 12 m)")
bad = q9[(q9["along_m"] < -5) | (q9["along_m"] > q9["line_m"] + 5)]
past = np.where(bad["along_m"] < 0, -bad["along_m"], bad["along_m"] - bad["line_m"])
add_flags(9, "quadrat_repeat", bad, "KEY", "quadrat is " + pd.Series(past, index=bad.index).round(1).astype(str) + " m past the end of the A to B line (limit 5 m)")

# ---------- 23. identical quadrat coordinates ----------
same_spot = quadrats.duplicated(["PARENT_KEY", "location_quadrat-Latitude", "location_quadrat-Longitude"], keep=False)
add_flags(23, "quadrat_repeat", quadrats[same_spot], "KEY", "same coordinates as another quadrat in this survey")


### Species checks (10 to 16)

Check 14: the SOP requires a species not on the list to be typed as `Genus_species`, with an underscore, and several names separated by a single space (Herbaceous SOP section 2 and section 5 step 7). In this data every typed name uses a space instead of an underscore, so every one is flagged. The form does not enforce the format. The spelling of the names is not checked, because there is no taxonomy list to check against.

In [36]:
# ---------- 10. species logic in each quadrat ----------
n_selected = quadrats["herb_species-selected_herb_species_uuids"].fillna("").str.split().str.len()
yes = quadrats["herbs_present"] == "yes"
extra_yes = quadrats["additional_species_present"] == "yes"
rows = {
    "herbs present but no species recorded": yes & (n_selected == 0) & ~extra_yes,
    "no herbs present but species recorded": ~yes & ((n_selected > 0) | extra_yes),
    "species count does not match the number selected": yes & quadrats["herb_species-count_herb_species"].notna() & (quadrats["herb_species-count_herb_species"] != n_selected),
    "all unknown but additional species is not yes": (quadrats["herb_species-all_unknown_unidentifiable"] == "yes") & ~extra_yes,
}
for text, mask in rows.items():
    add_flags(10, "quadrat_repeat", quadrats[mask], "KEY", text)

# ---------- 11. species is valid and herbaceous ----------
chosen = (quadrats[["KEY", "herb_species-selected_herb_species_uuids"]].dropna()
          .assign(uuid=lambda d: d["herb_species-selected_herb_species_uuids"].str.split()).explode("uuid")
          .merge(species_keep[["__id", "groups"]], left_on="uuid", right_on="__id", how="left"))
add_flags(11, "quadrat_repeat", chosen[chosen["__id"].isna()], "KEY", "species is not on the species list")
add_flags(11, "quadrat_repeat", chosen[chosen["__id"].notna() & (chosen["groups"] != "herbaceous")], "KEY", "species is not herbaceous")

# ---------- 12. plots with no species ----------
bad = survey[survey["survey_end-quadrats_with_species"] == 0]
add_flags(12, "herbaceous_veg_survey", bad, "KEY", "no quadrat has any species")

# ---------- 13. additional species: name is present ----------
add_flags(13, "additional_species_repeat", extra[extra["validated_name"].isna()], "KEY", "validated_name is empty")

# ---------- 14. name format ----------
canon = extra.loc[extra["species_entry_mode"] == "new_missing", ["KEY", "new_missing_canonical"]].copy()
one_name = r"[A-Z][a-z-]+_[a-z-]+"
well_formed = canon["new_missing_canonical"].str.fullmatch(one_name + "( " + one_name + ")*")
bad = canon[~well_formed.fillna(False)]
add_flags(14, "additional_species_repeat", bad, "KEY", "not in Genus_species format: '" + bad["new_missing_canonical"].astype(str) + "'")

# ---------- 15. a 'missing' species that is already on the species list ----------
on_list = set(species_keep["label"].str.strip().str.lower())
canon["clean"] = canon["new_missing_canonical"].str.strip().str.replace("_", " ").str.lower()
bad = canon[canon["clean"].isin(on_list)]
add_flags(15, "additional_species_repeat", bad, "KEY", "typed as missing but is on the species list")

# ---------- 16. photos, and reused records exist and are active ----------
new_unknown = extra["species_entry_mode"] == "new_unknown"
add_flags(16, "additional_species_repeat", extra[new_unknown & extra["macro_images-macro_1"].isna()], "KEY", "new unknown has no close-up photo")
active_extra = set(species_extra_keep.loc[species_extra_keep["choice_status"] == "active", "__id"])
all_extra = set(species_extra_keep["__id"])
for col in ["select_reuse_unknown", "select_reuse_missing"]:
    used = extra[extra[col].notna()]
    add_flags(16, "additional_species_repeat", used[~used[col].isin(all_extra)], "KEY", col + " is not in species_extra")
    add_flags(16, "additional_species_repeat", used[used[col].isin(all_extra) & ~used[col].isin(active_extra)], "KEY", col + " is not active")

### Plot registration checks (17 to 21)

A plot is the belt transect: 50 m long and 5 m wide, defined by a midpoint and endpoints A and B (Plot Registration SOP section 2). The 20 quadrats are the sampling units placed along it.

**Check 18 (transect length).** The SOPs say the transect is 50 m long, with each endpoint 25 m from the midpoint. They give no tolerance and say nothing about GPS error on the endpoints. The only statement on GPS error is Herbaceous SOP section 4.2, which asks for accuracy of 5 m or better.

Each GPS point in the form records an accuracy value. It is the device's own estimate of how far off the point could be, in meters, and the form requires it to be 5 m or better. A is recorded with accuracy 3 m, for example, and B with 4 m.

The rule treats the A to B distance as uncertain by the sum of those two numbers:

- With A at 3 m and B at 4 m, the A to B length could be off by up to 7 m. The allowed range is 50 ± 7, which is 43 to 57 m.
- If A and B both have 5 m accuracy, the range is 50 ± 10, which is 40 to 60 m.
- If both have 1 m accuracy, the range is 50 ± 2, which is 48 to 52 m.

Any plot whose measured A to B length falls outside its own range is flagged. A plot with poor GPS fixes gets a wider allowance. A plot with good fixes gets a stricter one, so a 45 m reading with 1 m accuracy points to a real problem, while the same reading with 5 m accuracy might just be GPS noise. A fixed range of 40 to 60 m would be the same rule assuming every point has the worst allowed accuracy of 5 m, which is too forgiving for plots with good fixes.

The same rule is applied to the A to midpoint and midpoint to B legs, which should each be 25 m. The transect is laid with a taut tape, so its true length is 50 m by construction. A flag therefore means either a GPS problem or a laying problem, and GPS alone cannot tell which. The tolerance is derived from the 5 m accuracy requirement, not stated in the SOPs.

In [37]:
# 17. midpoint within 25 m of the planned point
bad = reg[has_points & (reg["midpoint_to_planned_m"] > 25)]
add_flags(17, "register_vegetation_plots", bad, "KEY", "midpoint is " + bad["midpoint_to_planned_m"].round(1).astype(str) + " m from the planned point")

# 18. transect length: A to B should be 50 m, A to midpoint and midpoint to B 25 m, within the recorded accuracies
acc = {p: reg[f"plot_metadata-collect_points-{p}-Accuracy"] for p in ["endpoint_a-location_endpoint_a", "midpoint-location_midpoint", "endpoint_b-location_endpoint_b"]}
acc_a, acc_m, acc_b = acc.values()
for label, measured, nominal, tolerance in [("A to B", reg["a_to_b_m"], 50, acc_a + acc_b),
                                            ("A to midpoint", reg["a_to_mid_m"], 25, acc_a + acc_m),
                                            ("midpoint to B", reg["mid_to_b_m"], 25, acc_m + acc_b)]:
    bad = reg[has_points & ((measured - nominal).abs() > tolerance)]
    detail = (label + " is " + measured[bad.index].round(1).astype(str) + " m, expected " + str(nominal)
              + " m within " + tolerance[bad.index].round(1).astype(str) + " m")
    add_flags(18, "register_vegetation_plots", bad, "KEY", detail)

# 19. registration GPS accuracy 5 m or better
for point in ["endpoint_a-location_endpoint_a", "midpoint-location_midpoint", "endpoint_b-location_endpoint_b"]:
    col = f"plot_metadata-collect_points-{point}-Accuracy"
    bad = reg[has_points & ((reg[col] > 5) | reg[col].isna())]
    add_flags(19, "register_vegetation_plots", bad, "KEY", point.split("-")[0] + " accuracy " + bad[col].astype(str) + " m")

# 20. non-viable plots have a reason and were not surveyed
not_viable_reg = reg["plot_selection-is_plot_viable"] == "no"
add_flags(20, "register_vegetation_plots", reg[not_viable_reg & reg["plot_selection-plot_establishment_notes"].isna()], "KEY", "non-viable plot has no reason")
surveyed_centroids = set(vegplots_keep.loc[vegplots_keep["__id"].isin(survey[plot_col]), "plot_uuid"])
add_flags(20, "register_vegetation_plots", reg[not_viable_reg & reg["plot_selection-selected_plot_uuid"].isin(surveyed_centroids)], "KEY", "non-viable plot has a survey")

# 21. fire on a surveyed plot
fire = reg["plot_metadata-disturbance_types"].fillna("").str.split().apply(lambda x: "fire" in x)
add_flags(21, "register_vegetation_plots", reg[fire & reg["plot_selection-selected_plot_uuid"].isin(surveyed_centroids)], "KEY", "fire disturbance on a surveyed plot")

### Planned, registered and surveyed (22) and summary of flags

In [38]:
# ---------- 22. planned, registered and surveyed ----------
planned_summary = pd.DataFrame({
    "planned (centroids)": centroids_keep["sample_status"].value_counts(),
    "registered (vegplots)": vegplots_keep["plot_status"].value_counts(),
    "surveyed (distinct plots)": vegplots_keep[vegplots_keep["__id"].isin(survey[plot_col])]["plot_status"].value_counts(),
}).fillna(0).astype(int)
planned_summary.loc["total"] = planned_summary.sum()

flags_table = pd.concat(flag_tables, ignore_index=True)

flag_summary = (planned_checks.loc[planned_checks["id"] != 22, ["id", "level", "check", "priority"]]
                .merge(flags_table.groupby("check_id").agg(flags=("key", "size"), records=("key", "nunique")),
                       left_on="id", right_index=True, how="left")
                .fillna({"flags": 0, "records": 0})
                .astype({"flags": int, "records": int}))
flag_summary

,id,level,check,priority,flags,records
0,1,Survey,20 quadrats per survey,M,0,0
1,2,Survey,"Quadrat numbers 1 to 20, each once",M,0,0
2,3,Survey,"Plot is registered, viable and in the herbs su...",M,0,0
3,4,Survey,Repeated plot visits,M,4,4
4,5,Survey,Recorder and team are valid,S,0,0
5,6,Survey,Timing,S,0,0
6,7,Survey,Missing photos,O,0,0
7,8,Quadrat,GPS accuracy,M,0,0
8,9,Quadrat,Location is on the transect,S,193,147
9,10,Quadrat,Species logic,M,0,0
